**GLB_RPA_OrchestratorOnPrem**

- **_Description:_** This notebook read csv files extracted from orchestrator and performs ETL to create a file to be consumed into Global RPA dashboards
- **_Developer:_** Edgard Pupo
- **_Last Update:_** Mar 10th, 2025

##### 1.0 Initialization
> ###### Read configuration files and initialize connections

In [0]:
# Import additional libraries exclusive for this notebook
from datetime import datetime, timezone, timedelta

# Set timezone to GMT-3
gmt = timezone(timedelta(hours=-3))

# Set start time
start_time = datetime.now(gmt)
print(f"Execution START: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")

# Display the notebook path                                                          
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
print(f"Current notebook path: {notebook_path}")

Execution START: 2026-01-07 07:05:27
Current notebook path: /SharedBusinessServices/SBS - Data Analytics and CIP/Data Analytics/02 - BUSINESS/RPA/GBL_RPA_OrchOnPrem


In [0]:
%run "/SharedBusinessServices/SBS - Data Analytics and CIP/Data Analytics/02 - BUSINESS/RPA/Orchestrator/GBL_RPA_GeneralConfigFile"

**GBL_ProjectConfigFile**

- **_Description:_** This notebook starts all projects variables, paths and credentials as needed
- **_Developer:_** Edgard Pupo
- **_Last Update:_** Mar 10th, 2025

2026-01-07 07:05:27 [BEGIN] Basic Config Sessions



##### 1.0 Initialization
> ###### Read configuration files and initialize connections

2026-01-07 07:05:27 [BEGIN] Basic Config Sessions


**GBL_BasicConfigSessions**

- **_Description:_** This notebook starts all basic configuration sessions between Databricks and data lake storage account
- **_Developer:_** Edgard Pupo
- **_Last Update:_** Mar 14th, 2025


...

##### 4.0 Constant Variables

2026-01-07 07:05:33 [BEGIN] Set constants variables
2026-01-07 07:05:33 [END]   Set constants variables


##### 5.0 Functions

2026-01-07 07:05:34 [BEGIN] Set functions variables


2026-01-07 07:05:34 [END]   Set functions variables

2026-01-07 07:05:34 [END] Basic Config Sessions


##### 2.0 Get Data
> ###### Get data to be processed

In [0]:
if PrdYear != "2025":
    dbutils.notebook.exit(f"Notebook skipped. YearPrd={PrdYear}")

In [0]:
## [DEVELOPMENT DB] Create a development db connections 'dev_df'                     

# Read csv file
#dev_df = sp.read_pd_from_csv(f"{DevRpaSharepointUrl}{DevFolderPath}{DevFilePath}", delimiter=";", low_memory=False, encoding="ISO-8859-1")

In [0]:
## [PRODUCTION DB] Create a development db connections 'prd_df'                     

# Read files in folder
prd_df = fxCSVFromFolder(PrdRpaSharepointUrl, PrdFolderPath, PrdYearFolder)

In [0]:
# Set enviroment db to be used in the project
df = prd_df

In [0]:
# Convert pandas to spark dataframe
spark_df = (
    spark.createDataFrame(df, fxInferSchema(df.columns.tolist()))
    .filter(f.col("Status").isNotNull() & f.col("Type").isNotNull())
    .withColumn(
        "Type",
        f.when(f.col("Type").isNull() & f.col("Status").isNotNull(), "RUN")
         .otherwise(f.upper(f.col("Type"))),
    )
)

run_df = (
  spark_df
  .filter(spark_df['Type'] == 'RUN')
  .withColumn("Date", f.to_date(f.to_timestamp(f.col("Start"), "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"), "yyyy-MM-dd"))
)

transaction_df = (
  spark_df
  .filter(spark_df['Type'] == 'TRANSACTION')
  .withColumn("Date", f.to_date(f.to_timestamp(f.col("Start"), "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"), "yyyy-MM-dd"))
)

# Display filter results
sparkcount    = spark_df.count()
run           = run_df.count()
transaction   = transaction_df.count()
print(f"Rows ALL: {format(sparkcount, ',')} | RUN: {format(run, ',')} | TRANSACTIONS: {format(transaction, ',')}")

##### 3.0 Process
> ###### Process data modification (ETL)

###### 3.1 For RUN data type

In [0]:
# Join df to get a new df with naming convention applied
df_joined = run_df.join(
    namingconvention_df,
    run_df["Queue"] == namingconvention_df["Description"],
    how = "left"
)

# Replace values on 'Queue' column basis n convention
df_replaced = df_joined.withColumn(
    "Queue",
    f.coalesce(
      f.col("Code"), 
      f.col("Queue")
      )
)

# Remove temp columns
run_df_nameconv = (
    df_replaced
    .drop("Description", "Code", "Type")
    .filter(f.col("Queue").startswith("RPA"))
    .withColumn("Global ID", f.expr("substring(Queue, 1, 7)"))
    .filter(f.col("Global ID").startswith("RPA") & ~f.col("Global ID").startswith("RPAGP"))
)

In [0]:
# Filter 'Queue'basis on timeline
filtered_run_df = (
    run_df_nameconv.alias("run")
    .join(timeline_df.alias("timeline"), run_df_nameconv["Global ID"] == timeline_df["Queue"], how="left")
    .filter(
        (f.col("timeline.1stTransaction_Date").isNull()) | (f.col("run.Date") < f.col("timeline.1stTransaction_Date"))
    )
    .select("run.*")
    .drop("Global ID", "Date")
)

# Display filter results
before = run
run    = filtered_run_df.count()
diff   = before - run
print(f"Rows BEFORE: {format(before, ',')} | NOW: {format(run, ',')} | REMOVED: {format(diff, ',')}")

In [0]:
# Create a "Success" keywords list
success_keywords = ["SUCCESSFUL"]

# Replace Exception values basis keywords listed
run_df_exception = filtered_run_df.withColumn(
    "Exception",
    f.when(f.upper(f.col("Status")).isin([word.upper() for word in success_keywords]), "Success")
     .otherwise("Application")
)

###### 3.2 For TRANSACTIONS data type

In [0]:
# Define a short df name
df_filtered = transaction_df.where("""
    TRIM(UPPER(Folder)) <> 'TRANSACTION' AND
    Folder IS NOT NULL AND TRIM(Folder) <> '' AND

    (Details IS NULL OR UPPER(Details) <> 'VALIDATION') AND
    (Details IS NULL OR UPPER(Details) NOT LIKE '%ISCONSIDER%') AND

    UPPER(TRIM(Queue)) LIKE 'RPA%' AND
    UPPER(Queue) NOT LIKE '%TEST%' AND

    UPPER(Status) IN ('FAILED', 'RETRIED', 'SUCCESSFUL')
""")

# Display filter results
before = transaction
after  = df_filtered.count()
diff   = before - after
print(f"Rows BEFORE: {format(before, ',')} | NOW: {format(after, ',')} | REMOVED: {format(diff, ',')}")

In [0]:
# Create a data column to be used to filter out decommissioned queues
transaction_df = df_filtered.withColumn("Date", f.to_timestamp(df_filtered["Start"], "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"))
transaction_df = transaction_df.withColumn("Date", f.to_date(transaction_df["Date"], "dd/MM/yyyy"))

# Change 'Decommissioned_Date' schema to date
offqueueslist_df = offqueueslist_df.withColumn("Decommissioned_Date", f.to_date(offqueueslist_df["Decommissioned_Date"], "dd/MM/yyyy"))

# Join 'offqueuelist_df' to 'transaction_df' to get decomissioned queues dates
df_joined = transaction_df.join(
    offqueueslist_df,
    transaction_df["Queue"] == offqueueslist_df["Decommissioned_Queue"],
    how = "left"
)

# Remove queues decommissioned
df_joined = df_joined.where("""
  Decommissioned_Date > Date OR
  Decommissioned_Date IS NULL
  """
)

# Remove joined columns
transaction_df = df_joined.drop("Type", "Date", "Full_Orchestrator_Process_Name", "Decommissioned_Queue", "Decommissioned_Date")

# Display filter results
before = transaction
transaction  = transaction_df.count()
diff   = before - transaction
print(f"Rows BEFORE: {format(before, ',')} | NOW: {format(transaction, ',')} | REMOVED: {format(diff, ',')}")

In [0]:
# Join df to get a new df with naming convention applied
df_joined = None
df_joined = transaction_df.join(
    namingconvention_df,
    transaction_df["Queue"] == namingconvention_df["Description"],
    how = "left"
)

# Replace values on 'Queue' column basis n convention
df_replaced = df_joined.withColumn(
    "Queue",
    f.coalesce(
      f.col("Code"), 
      f.col("Queue")
      )
)

# Remove temp columns
transaction_df_nameconv = df_replaced.drop("Description", "Code")

In [0]:
# Replace Exception values basis keywords listed
transaction_df_exception = transaction_df_nameconv.withColumn(
    "Exception",
    f.when(f.col("Exception") == "BusinessException", "Business")
     .when(f.col("Exception") == "ApplicationException", "Application")
     .otherwise("Success")
)

###### 3.3 For ALL data type

In [0]:
# Append both dfs after all specific ETL done
appended_df = run_df_exception.union(transaction_df_exception)

# Rename columns
appended_df = appended_df.withColumnRenamed("end", "End")

# Display filter results
before = run
after  = transaction
diff  = before + after
print(f"RUN: {format(before, ',')} | TRANSACTION: {format(after, ',')} | TOTAL: {format(diff, ',')}")

In [0]:
# Replace values on 'Folder' column basis current values loaded from orchestrator
replaced_df = appended_df.withColumn(
    "Folder",
    f.when(f.col("Folder") == "ARG", "SWLT")
    .when(f.col("Folder") == "ARG-LDC-PRD", "SWLT")
    .when(f.col("Folder") == "CHINA", "CHN")
    .when(f.col("Folder") == "EMEA", "EMEA")
    .when(f.col("Folder") == "INDIA", "IND")
    .when(f.col("Folder") == "SINGAPORE", "SGP")
    .when(f.col("Folder") == "SOFIA", "SOFIA")
    .when(f.col("Folder") == "GDOC", "GDOC")
    .when(f.col("Folder") == "NAM", "NAM")
    
    .otherwise(
        # after checking values on 'Folder' column, check 'Orch' column
        f.when(f.col("Orch") == "NAM", "NAM")
        .when(f.col("Orch") == "NAM_CoE", "NAM")
        .when(f.col("Orch") == "SWLT", "SWLT")
        .when(f.col("Orch") == "NLT", "NLT")
        .when(f.col("Orch") == "IT", "EMEA")
        .when(f.col("Orch") == "IND", "IND")
        
        .otherwise(
            f.when(f.col("Queue") == "RPA0231_Global_EPM_Files_Refresh_REPORTING", "EMEA")
            
            .otherwise("NLT")
            )
    )
)

# Remove uneeded columns
replaced_df = replaced_df.drop("Orch")

In [0]:
# Create a new column basis on 'Folder' column
macroregion_df = replaced_df.withColumn(
    "Macro Region",
    f.when(
        (f.col("Folder") == "NAM") |
        (f.col("Folder") == "NLT") |
        (f.col("Folder") == "SWLT"),
        "AMERICAS",
    ).otherwise("EUROASIA"),
)

In [0]:
# Create a new colum basis on 'Queue' column
globalid_df = (
    macroregion_df.withColumn("Global ID", f.col("Queue").substr(1, 7))
    .withColumn("Details", f.when(f.col("Details") == "nan", "").otherwise(f.col("Details")))
    .fillna({"Details": ""})
)

# Remove special characters from columns names
globalid_df = globalid_df.toDF(
    *[
        col_name.replace(" ", "_")
        .replace(";", "")
        .replace("{", "")
        .replace("}", "")
        .replace("(", "")
        .replace(")", "")
        .replace("\n", "")
        .replace("\t", "")
        .replace("=", "")
        for col_name in globalid_df.columns
    ]
)

In [0]:
# Reorder columns in the DataFrame
columnslist_df = ["Macro_Region", "Folder", "Global_ID", "Reference", "Start", "End", "Status","Exception", "ExceptionReason"]
reorderded_df  = globalid_df.select(*columnslist_df)

# Rename columns and change date/time format to fit dashboard legacy structure
adjusted_df = (
  reorderded_df
  .withColumnRenamed("Global_ID", "Queue")
  .withColumn("Start", f.date_format(f.col("Start"), "dd/MM/yyyy HH:mm:ss"))
  .withColumn("End", f.date_format(f.col("End"), "dd/MM/yyyy HH:mm:ss"))
)

# Create a Global ID list for replacement
globalidlist = {
    "RPA0010": "RPA0009",
    "RPA0029": "RPA0028",
    "RPA0037": "RPA0036",
    "RPA0046": "RPA0045",
    "RPA0051": "RPA0050",
    "RPA0066": "RPA0065",
    "RPA0074": "RPA0073",
    "RPA0089": "RPA0088",
    "RPA0110": "RPA0109",
    "RPA0235": "RPA0234",
}

# Dictionary conversion to fit create_map()
queue_mapping = f.create_map([f.lit(x) for pair in globalidlist.items() for x in pair])

# Replace "Queue" values
replacedqueues_df = (
    adjusted_df
    .withColumn("Queue", f.coalesce(queue_mapping[f.col("Queue")], f.col("Queue")))
    .withColumn("Status", f.when(adjusted_df["Status"] == "Failed", "Faulted").otherwise(adjusted_df["Status"]))
)

removedrows_df = replacedqueues_df.filter(f.col("Status") != "Stopped")

In [0]:
'''
test_df = removedrows_df.withColumn("Start", f.to_timestamp(removedrows_df["Start"], "dd/MM/yyyy HH:mm:ss"))
test_df = test_df.withColumn("Start", f.to_date(f.col("Start")))

robots = ["RPA0009", "RPA0010", "RPA0017", "RPA0028", "RPA0029", "RPA0034", "RPA0044", "RPA0048", "RPA0095", "RPA0125", "RPA0162"]

previous_start_period = "2025-04-03"
previous_end_period   = "2025-04-09"

current_start_period  = "2025-04-10"
current_end_period    = "2025-04-16"

print("Previous Period:")
display(
  test_df
    .select("Queue", "Start")
    .filter((f.col("Start") >= previous_start_period) & (f.col("Start") <= previous_end_period))
    .filter(f.col("Queue").isin(robots))
    .select("Queue")
    .distinct()
    .orderBy("Queue", "Start")
  )

print("Current Period:")
display(
  test_df
    .select("Queue", "Start")
    .filter((f.col("Start") >= current_start_period) & (f.col("Start") <= current_end_period))
    .filter(f.col("Queue").isin(robots))
    .select("Queue")
    .distinct()
    .orderBy("Queue", "Start")
  )
'''

##### 4.0 End Process
> ###### Export results to data lake

In [0]:
# Save a parquet file into a folder in the data lake
final_df = removedrows_df
final_df.write.mode('overwrite').parquet(f"abfss://rep@{dl.adls_client.primary_hostname}{parquetsavepath}{parquetfilename}")

In [0]:
# Define db and table names
db_table = parquetfilename.split(".")[0]

# Delete table if exists on DB
# spark.sql(f"DROP TABLE IF EXISTS `{db_name}`.`{db_table}`")

# Overewrite the spark df to a table on Hive-Metastore
final_df.write.mode("overwrite").saveAsTable(f"{db_name}.{db_table}")

In [0]:
# Set end time
end_time = datetime.now(gmt)

# Calculate execution time and format removing microseconds
execution_time = end_time - start_time
execution_time_str = str(execution_time).split(".")[0]

# Calculate ttl of rows
final_sparkcount = final_df.count()

# Print results details
print("RESULTS REPORT:")
print(f"{PrdYear} year processed")
print(f"Execution START: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Execution END  : {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Execution time : {execution_time_str}")
print('')
print(f"Ttl nbr of rows at START  :  {format(sparkcount, ',')}")
print(f"Ttl nbr of rows in the END:  {format(final_sparkcount, ',')}")